# 측정도구 신뢰도

척도 정의와 결측값 검증은 공통 모듈을 사용합니다. Cronbach α와 95% 신뢰구간을 한 번 계산해 최종표와 그림에서 재사용합니다.

In [ ]:
from pathlib import Path
import sys

A2UI_DIR = Path.cwd() if (Path.cwd() / "analysis_common.py").exists() else Path.cwd() / "Data" / "A2UI"
if str(A2UI_DIR) not in sys.path:
    sys.path.insert(0, str(A2UI_DIR))
import analysis_common as common
from functools import partial
from analysis_common import significance_stars, format_p

import numpy as np
import pandas as pd
import pingouin as pg
from IPython.display import display
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

KOREAN_FONT = common.configure_plots()
FIGURE_DIR = common.FIGURE_DIR


## 1. 원자료와 문항 확인

In [ ]:
survey = common.load_survey()
CONSTRUCT_ITEMS = common.CONSTRUCT_ITEMS
print(f"자료 크기: {survey.shape[0]}명 × {survey.shape[1]}개 열")
display(pd.DataFrame({"문항 번호": {name: ", ".join(items) for name, items in CONSTRUCT_ITEMS.items()}}))


## 2. 신뢰도 계산

α≥.70을 충족 기준으로 사용합니다. 단일 문항의 α와 신뢰구간은 계산하지 않습니다.

In [ ]:
rows = []
for construct, prefixes in CONSTRUCT_ITEMS.items():
    items = common.item_scores(survey, prefixes)
    alpha, ci = pg.cronbach_alpha(items, nan_policy="listwise") if items.shape[1] > 1 else (np.nan, None)
    rows.append({"측정변수": construct, "문항 수": items.shape[1], "N": len(items), "Cronbach's α": alpha,
                 "95% CI": "—" if ci is None else f"[{ci[0]:.3f}, {ci[1]:.3f}]"})
reliability_results = pd.DataFrame(rows)
publication_reliability_table = reliability_results.copy()
publication_reliability_table["판정"] = publication_reliability_table["Cronbach's α"].map(
    lambda x: "단일 문항" if pd.isna(x) else "충족" if x >= .70 else "미달")
publication_reliability_table["Cronbach's α"] = publication_reliability_table["Cronbach's α"].map(
    lambda x: "—" if pd.isna(x) else f"{x:.3f}")
display(publication_reliability_table.set_index("측정변수"))


## 3. 신뢰도 그림

In [ ]:
plot_data = reliability_results.dropna(subset=["Cronbach's α"]).copy()
plot_data = plot_data.sort_values("Cronbach's α")
y_positions = np.arange(len(plot_data))
alpha_values = plot_data["Cronbach's α"].to_numpy()

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.hlines(y_positions, .70, alpha_values, color="0.60", linewidth=1.8)
ax.scatter(alpha_values, y_positions, color="black", s=42, zorder=3)
ax.axvline(.70, color="0.25", linestyle="--", linewidth=1.2, label="기준 α=.70")

for y_position, alpha_value in zip(y_positions, alpha_values):
    ax.text(alpha_value + .006, y_position, f"{alpha_value:.3f}", va="center", fontsize=9)

ax.set_yticks(y_positions, plot_data["측정변수"])
ax.set_xlim(.65, 1.00)
ax.set_xlabel("Cronbach's α")
ax.set_ylabel("측정변수")
ax.set_title("사용자실험 측정도구의 신뢰도")
ax.grid(axis="x", color="0.88", linewidth=.7)
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

fig.savefig(FIGURE_DIR / "figure_01_reliability_alpha.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure_01_reliability_alpha.pdf", bbox_inches="tight")
plt.show()
